# SnapUGC-LightKD Opening-5s 5000 Run

This notebook re-extracts the same 5000-video ECR-balanced subset, but samples visual and motion features from the first 5 seconds. This follows the SnapUGC/ECR papers: ECR is `P(watch > 5s)`, so the opening moments should be more important than uniformly sampling the whole video.

Main changes versus the old notebook:

- `extract_features.py --opening-seconds 5`
- train Teacher `v8_rich_inputs`, which uses `yamnet_class_probs` and numeric metadata in addition to the original modalities
- keep the same subset seed `42` and ECR-balanced 5000 split


In [ ]:
# 0. Clone this repository into Kaggle working directory
import os, sys, json, glob, shutil, subprocess, textwrap
from pathlib import Path

REPO_URL = 'https://github.com/TranTop2806/SnapUGC-LightKD.git'
REPO_DIR = '/kaggle/working/SnapUGC-LightKD'

if not os.path.exists(REPO_DIR):
    %cd /kaggle/working
    !git clone -q "$REPO_URL" "$REPO_DIR"
else:
    print('Repo exists:', REPO_DIR)

%cd "$REPO_DIR"
!git status --short


In [ ]:
# 1. Install dependencies
# Kaggle usually already has torch/torchvision. These packages cover extraction utilities.
!pip install -q decord transformers sentence-transformers pillow tqdm pandas scipy scikit-learn tensorflow tensorflow-hub tf-keras

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


In [ ]:
# 2. Paths and ECR-balanced 5000-video subset
import os, sys, glob, json, subprocess
import pandas as pd
import torch

MAX_VIDEOS = 5000
SUBSET_SEED = 42
ECR_BINS = 10
OPENING_SECONDS = 5
RUN_DOVER = True
RUN_CAPTION = False  # Smoke-run first. Set True later only if you need BLIP captions.
RESET_FEATURES = True
CAPTION_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

INPUT_DIR = '/kaggle/input/datasets/nguyntuncng/snapugc-dataset'
TRAIN_CSV_ORIG = os.path.join(INPUT_DIR, 'train_data.csv')
TRAIN_VIDEO_ROOT = os.path.join(INPUT_DIR, 'train_videos', 'train_videos')
if not os.path.exists(TRAIN_CSV_ORIG):
    raise FileNotFoundError(TRAIN_CSV_ORIG)
if not os.path.isdir(TRAIN_VIDEO_ROOT):
    raise FileNotFoundError(TRAIN_VIDEO_ROOT)

OUTPUT_DIR = f'/kaggle/working/opening{OPENING_SECONDS}_final_{MAX_VIDEOS}_videos'
SUBSET_VIDEO_DIR = os.path.join(OUTPUT_DIR, 'subset_videos')
SUBSET_CSV = os.path.join(OUTPUT_DIR, f'train_subset_{MAX_VIDEOS}.csv')
FEATURES_PATH = os.path.join(OUTPUT_DIR, f'features_opening{OPENING_SECONDS}_{MAX_VIDEOS}.json')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
TEACHER_DIR = os.path.join(OUTPUT_DIR, 'teacher_experiments')
DOVER_DIR = os.path.join(OUTPUT_DIR, 'dover')
DOVER_CSV = os.path.join(DOVER_DIR, 'dover_scores.csv')
for d in [OUTPUT_DIR, RESULTS_DIR, TEACHER_DIR, DOVER_DIR]:
    os.makedirs(d, exist_ok=True)

subset_cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/make_subset.py'),
    '--csv', TRAIN_CSV_ORIG,
    '--videos', TRAIN_VIDEO_ROOT,
    '--out-csv', SUBSET_CSV,
    '--out-videos', SUBSET_VIDEO_DIR,
    '--max', str(MAX_VIDEOS),
    '--seed', str(SUBSET_SEED),
    '--bins', str(ECR_BINS),
]
if RESET_FEATURES:
    subset_cmd.append('--reset')
print(' '.join(subset_cmd))
subprocess.run(subset_cmd, check=True)

if RESET_FEATURES:
    for stale_path in [FEATURES_PATH, DOVER_CSV]:
        if os.path.exists(stale_path):
            os.remove(stale_path)

subset_df = pd.read_csv(SUBSET_CSV)
ecr = pd.to_numeric(subset_df['ECR'], errors='coerce')
print('Subset rows:', len(subset_df))
print('Subset videos:', len(glob.glob(os.path.join(SUBSET_VIDEO_DIR, '*.mp4'))))
print(ecr.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))
print('Fixed 0.1-bin counts:')
print(pd.cut(ecr, bins=[0,.1,.2,.3,.4,.5,.6,.7,.8,.9,1.0], include_lowest=True).value_counts().sort_index())
print('Quantile counts:')
print(pd.qcut(ecr, q=min(ECR_BINS, ecr.nunique(), len(ecr)), duplicates='drop').value_counts().sort_index())
print('FEATURES_PATH:', FEATURES_PATH)


In [ ]:
# 3. DOVER-Mobile quality scores for the bounded subset
if RUN_DOVER:
    !git clone -q https://github.com/VQAssessment/DOVER.git /kaggle/working/DOVER 2>/dev/null || true
    %cd /kaggle/working/DOVER
    !pip install -q scikit-video yacs timm einops opencv-python-headless decord pyyaml pandas scipy tqdm
    !mkdir -p pretrained_weights
    !wget -q -nc https://github.com/QualityAssessment/DOVER/releases/download/v0.5.0/DOVER-Mobile.pth -O pretrained_weights/DOVER-Mobile.pth
    !python evaluate_a_set_of_videos.py -in "$SUBSET_VIDEO_DIR" -out "$DOVER_CSV" -o dover-mobile.yml
    if not os.path.exists(DOVER_CSV):
        raise RuntimeError(f'DOVER did not produce expected CSV: {DOVER_CSV}')
    dover_preview = pd.read_csv(DOVER_CSV)
    print('DOVER rows:', len(dover_preview))
    print(dover_preview.head())
    if len(dover_preview) < max(1, int(MAX_VIDEOS * 0.8)):
        raise RuntimeError(f'DOVER produced too few rows: {len(dover_preview)}/{MAX_VIDEOS}')
else:
    raise RuntimeError('RUN_DOVER must be True for this run.')

%cd /kaggle/working
print('DOVER_CSV:', DOVER_CSV)


In [ ]:
# 4. Opening-5s feature extraction with BLIP-base captions

# Kill stale extraction processes before starting. This prevents two heavy model-loaders
# from fighting for the same GPU/cache if the Kaggle cell was interrupted/re-run.
!pkill -9 -f scripts/extract_features.py 2>/dev/null || true
!ps -ef | grep extract_features.py | grep -v grep || true

cmd = [
    sys.executable, '-u', os.path.join(REPO_DIR, 'scripts/extract_features.py'),
    '--csv', SUBSET_CSV,
    '--videos', SUBSET_VIDEO_DIR,
    '--out', FEATURES_PATH,
    '--max', str(MAX_VIDEOS),
    '--save-every', '50',
    '--num-frames', '16',
    '--motion-clips', '4',
    '--motion-frames', '16',
    '--opening-seconds', str(OPENING_SECONDS),
    '--caption-frames', '3',
    '--caption-device', CAPTION_DEVICE,
    '--text-device', 'cpu',
    '--lock-file', os.path.join(OUTPUT_DIR, 'extract_features.lock'),
    '--max-errors-before-abort', '1',
]
if DOVER_CSV:
    cmd += ['--dover-csv', DOVER_CSV]
if RUN_CAPTION:
    cmd += ['--strict-caption']
else:
    cmd += ['--skip-caption']
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 5. Validate opening feature schema
with open(FEATURES_PATH, 'r', encoding='utf-8') as f:
    features = json.load(f)
print('samples:', len(features))
if len(features) != MAX_VIDEOS:
    raise RuntimeError(f'Expected exactly {MAX_VIDEOS} successful samples, got {len(features)}')

caption_ok = sum(1 for x in features if str(x.get('blip_caption') or x.get('caption') or '').strip())
dover_ok = sum(1 for x in features if bool((x.get('dover_scores') or {}).get('found', False)))
window_ok = sum(1 for x in features if float(x.get('feature_window_seconds') or 0) <= OPENING_SECONDS + 1e-6)
print(f'BLIP non-empty captions: {caption_ok}/{len(features)}')
print(f'DOVER matched scores: {dover_ok}/{len(features)}')
print(f'Opening-window metadata ok: {window_ok}/{len(features)}')
if RUN_CAPTION and caption_ok != len(features):
    raise RuntimeError(f'Every sample must have a BLIP caption, got {caption_ok}/{len(features)}')
if RUN_DOVER and dover_ok != len(features):
    raise RuntimeError(f'Every sample must have a matched DOVER score, got {dover_ok}/{len(features)}')

first = features[0]
for key in ['duration', 'feature_window_seconds', 'feature_num_frames', 'clip_frame_embeddings', 'motion_clip_embeddings', 'yamnet_class_probs', 'dover_scores', 'blip_caption']:
    value = first.get(key)
    if isinstance(value, list):
        print(key, 'len:', len(value), 'nested:', len(value[0]) if value and isinstance(value[0], list) else '')
    else:
        print(key, value)


In [ ]:
# 6. Train Teacher V8 rich-input on opening-5s features
# This is the current best local Teacher architecture after the 5000-run search.
teacher_cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/train_teacher_experiment.py'),
    '--data', FEATURES_PATH,
    '--save-dir', os.path.join(TEACHER_DIR, 'v8_rich_inputs_opening5_h384_d035_wd005_seed99'),
    '--model-version', 'v8_rich_inputs',
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--teacher-hidden', '384',
    '--teacher-heads', '6',
    '--teacher-blocks', '1',
    '--dropout', '0.35',
    '--aux-weight', '0.15',
    '--teacher-lr', '1e-4',
    '--weight-decay', '0.05',
    '--warmup-epochs', '5',
    '--teacher-epochs', '20',
    '--batch', '32',
    '--eval-batch', '128',
    '--seed', '99',
    '--split-seed', '42',
    '--selection-metric', 'final_score',
]
print(' '.join(teacher_cmd))
subprocess.run(teacher_cmd, check=True)


In [ ]:
# 7. Optional: train a few extra seeds for same-split ensemble
# Turn RUN_EXTRA_SEEDS on if the first V8 score is promising, e.g. >= 0.58.
RUN_EXTRA_SEEDS = False
extra_seeds = [314, 7, 2024]
if RUN_EXTRA_SEEDS:
    for seed in extra_seeds:
        save_dir = os.path.join(TEACHER_DIR, f'v8_rich_inputs_opening5_h384_d035_wd005_seed{seed}')
        cmd = teacher_cmd.copy()
        cmd[cmd.index('--save-dir') + 1] = save_dir
        cmd[cmd.index('--seed') + 1] = str(seed)
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)


In [ ]:
# 8. Compare Teacher experiments and package outputs
compare_cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/compare_teacher_experiments.py'),
    '--root', TEACHER_DIR,
]
print(' '.join(compare_cmd))
subprocess.run(compare_cmd, check=True)

comparison_csv = os.path.join(TEACHER_DIR, 'teacher_experiment_comparison.csv')
print(pd.read_csv(comparison_csv).head(20).to_string())


In [ ]:
# 9. Package outputs without subset videos
import zipfile
zip_path = f'/kaggle/working/snapugc_lightkd_opening{OPENING_SECONDS}_{MAX_VIDEOS}_outputs.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

package_files = [SUBSET_CSV, FEATURES_PATH, DOVER_CSV]
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in package_files:
        if path and os.path.exists(path):
            zf.write(path, arcname=os.path.relpath(path, OUTPUT_DIR))
    for root, _, files in os.walk(TEACHER_DIR):
        for name in files:
            path = os.path.join(root, name)
            zf.write(path, arcname=os.path.relpath(path, OUTPUT_DIR))

print(zip_path)
print('Excluded from zip:', SUBSET_VIDEO_DIR)
print('Zip size MB:', round(os.path.getsize(zip_path) / (1024 * 1024), 2))
